Tableau 1 : Ici nous créons une table descriptive présentant les caractéristiques de la population étudiée à l'inclusion (*baseline*), stratifiée par bras de traitement.


In [ ]:
#On commence par charger les différentes bibliothèques et les installer au préalable si ce n'est pas déjà fait
import pandas as pd
import polars as pl
import numpy as np
from tableone import TableOne
# Configuration d'affichage (optionnel mais pratique)
pl.Config.set_tbl_rows(20)
dm1 = pl.read_csv("dm1.csv")
dm2 = pl.read_csv("dm2.csv")
dm = pl.concat([dm1, dm2])
dm.columns

['STUDYID',
 'DOMAIN',
 'USUBJID',
 'EPOCH',
 'VISIT',
 'VISITNUM',
 'RFSTDTC',
 'RFENDTC',
 'SITEID',
 'BRTHDTC',
 'AGE',
 'AGEU',
 'SEX',
 'RACE',
 'ETHNIC',
 'ARMCD',
 'ARM',
 'COUNTRY',
 'DMDTC',
 'DMDY']

In [ ]:
dm.select(
    pl.col("USUBJID"),
    pl.col("AGE"),
    pl.col("SEX"),
    pl.col("RACE"),
    pl.col("ETHNIC"),
    pl.col("ARMCD"),
).filter(
    pl.col("AGE").str.len_chars() > 3, #Variable manquante pour la variable AGE, on adapte donc le script
)

dm.shape

(411, 20)

In [ ]:
dm = dm.with_columns(
    pl.when(pl.col("AGE").str.strip_chars() == "")
      .then(None)
      .otherwise(pl.col("AGE"))
      .alias("AGE")
).with_columns(
    pl.col("AGE").cast(pl.Float64, strict=False)
)



In [ ]:
# Conversion des chaînes de charactères en flottants.
dm = dm.select(
    pl.col('USUBJID'),
    pl.col('AGE').cast(pl.Float64),
    pl.col('SEX'),
    pl.col('RACE'),
    pl.col('ETHNIC'),
    pl.col('ARMCD'),
).head(
    n=2
)

In [ ]:
continuous = ['AGE'] # Variables continues
categorical = ['SEX', 'RACE', 'ETHNIC'] # Variable catégorielles
groupby = 'ARMCD'  # On stratifie par bras de traitement

In [ ]:
#Calcul des effectifs avec filtrage pour la clonidine
dm.filter(
    pl.col('ARMCD') == 'CLON',
).group_by(
    pl.col('SEX'),
    maintain_order=True,
).len(
)

SEX,len
str,u32
"""F""",2


In [ ]:
#On calcule par sexe ici
dm.filter(
    pl.col('ARMCD') == 'CLON',
).group_by(
    pl.col('SEX'),
    maintain_order=True,
).len(
).with_columns(
    percentage=pl.col('len') / pl.sum('len'),
)

SEX,len,percentage
str,u32,f64
"""F""",2,1.0


In [ ]:
# Calcule les effectifs et les pourcentages d'une variable catégorielle spécifique  en filtrant et en stratifiant les données par le bras de traitement donné
def cat_table(df, arm, cat_var):
    t = dm.filter(
        pl.col('ARMCD') == arm,
    ).group_by(
        pl.col(cat_var),
        maintain_order=True,
    ).len(
    ).with_columns(
        percentage=pl.col('len') / pl.sum('len'),
    )
    return t

In [ ]:
m_list = []
type(m_list)
bupn_cat = [cat_table(dm, arm='BUPNAL', cat_var=cat) for cat in categorical]
clon_cat = [cat_table(dm, arm='CLON', cat_var=cat) for cat in categorical]
clon_cat


[shape: (1, 3)
 ┌─────┬─────┬────────────┐
 │ SEX ┆ len ┆ percentage │
 │ --- ┆ --- ┆ ---        │
 │ str ┆ u32 ┆ f64        │
 ╞═════╪═════╪════════════╡
 │ F   ┆ 2   ┆ 1.0        │
 └─────┴─────┴────────────┘,
 shape: (2, 3)
 ┌─────────────────────────────────┬─────┬────────────┐
 │ RACE                            ┆ len ┆ percentage │
 │ ---                             ┆ --- ┆ ---        │
 │ str                             ┆ u32 ┆ f64        │
 ╞═════════════════════════════════╪═════╪════════════╡
 │ WHITE                           ┆ 1   ┆ 0.5        │
 │ BLACK, AFRICAN AMERICAN, OR NE… ┆ 1   ┆ 0.5        │
 └─────────────────────────────────┴─────┴────────────┘,
 shape: (1, 3)
 ┌────────┬─────┬────────────┐
 │ ETHNIC ┆ len ┆ percentage │
 │ ---    ┆ --- ┆ ---        │
 │ str    ┆ u32 ┆ f64        │
 ╞════════╪═════╪════════════╡
 │ null   ┆ 2   ┆ 1.0        │
 └────────┴─────┴────────────┘]

In [ ]:
effectifs = dm.group_by(
    pl.col('ARMCD'),
    maintain_order=True,
).len(
).with_columns(
    percentage=pl.col('len') / pl.sum('len'),
)

effectifs

ARMCD,len,percentage
str,u32,f64
"""CLON""",2,1.0


In [ ]:
cont = dm.group_by(
    pl.col('ARMCD'),
    maintain_order=True,
).agg(
    mean_age=pl.col('AGE').mean(),
    std_age=pl.col('AGE').std(),
    med_age=pl.col('AGE').median(),
)

cont

ARMCD,mean_age,std_age,med_age
str,f64,f64,f64
"""CLON""",34.355921,10.003068,34.355921


In [ ]:
cont.columns

['ARMCD', 'mean_age', 'std_age', 'med_age']

In [ ]:
cont = pd.DataFrame(cont, columns=cont.columns)
cont
cont.columns
cont.columns = pd.MultiIndex.from_product([['age'], cont.columns])
cont
effectifs = pd.DataFrame(effectifs, columns=['ARMCD', 'n', 'percentage'])
effectifs
effectifs.columns = pd.MultiIndex.from_product([['effectifs'], effectifs.columns])
effectifs
effectifs.merge(cont, left_index=True, right_index=True)
effectifs.merge(cont, left_index=True, right_index=True).transpose()
t = effectifs.merge(cont, left_index=True, right_index=True).transpose()
t = t.drop(index=('age', 'ARMCD'))
t

0
effectifs ARMCD            CLON
          n                   2
          percentage        1.0
age       mean_age    34.355921
          std_age     10.003068
          med_age     34.355921

In [ ]:
import polars as pl
mh1 = pl.read_csv("mh1.csv")
mh2 = pl.read_csv("mh2.csv")
mh = pl.concat([mh1, mh2])
mh.columns

['STUDYID',
 'DOMAIN',
 'USUBJID',
 'EPOCH',
 'MHSEQ',
 'MHSPID',
 'MHTERM',
 'MHOCCUR',
 'MHSTAT',
 'VISIT',
 'VISITNUM',
 'MHDTC',
 'MHDY',
 'MHENRF']

In [ ]:
mh.select(
    pl.col('VISITNUM'),
).unique()

VISITNUM
i64
1
0


In [ ]:
mh.select(
    pl.col('USUBJID'),
    pl.col('MHTERM'),
    pl.col('MHOCCUR'),
).head(
    n=2
)

USUBJID,MHTERM,MHOCCUR
str,str,str
"""01_000579""","""ALLERGIES""","""N"""
"""01_000579""","""ALLERGIES""","""N"""


In [ ]:
mh.join(
    mh,
    on="USUBJID"
).head(
    n=10
)

STUDYID,DOMAIN,USUBJID,EPOCH,MHSEQ,MHSPID,MHTERM,MHOCCUR,MHSTAT,VISIT,VISITNUM,MHDTC,MHDY,MHENRF,STUDYID_right,DOMAIN_right,EPOCH_right,MHSEQ_right,MHSPID_right,MHTERM_right,MHOCCUR_right,MHSTAT_right,VISIT_right,VISITNUM_right,MHDTC_right,MHDY_right,MHENRF_right
str,str,str,str,i64,i64,str,str,str,str,i64,i64,i64,str,str,str,str,i64,i64,str,str,str,str,i64,i64,i64,str
"""NIDA-CTN-0001""","""MH""","""01_000579""","""SCREENING""",31,16,"""ALLERGIES""","""N""",null,"""BASELINE, ANY DAY PRIOR TO FIR…",0,2001,1,"""BEFORE""","""NIDA-CTN-0001""","""MH""","""SCREENING""",31,16,"""ALLERGIES""","""N""",null,"""BASELINE, ANY DAY PRIOR TO FIR…",0,2001,1,"""BEFORE"""
"""NIDA-CTN-0001""","""MH""","""01_000579""","""SCREENING""",32,16,"""ALLERGIES""","""N""",null,"""BASELINE, ANY DAY PRIOR TO FIR…",0,2001,1,"""DURING/AFTER""","""NIDA-CTN-0001""","""MH""","""SCREENING""",31,16,"""ALLERGIES""","""N""",null,"""BASELINE, ANY DAY PRIOR TO FIR…",0,2001,1,"""BEFORE"""
"""NIDA-CTN-0001""","""MH""","""01_000579""","""SCREENING""",5,3,"""CARDIOVASCULAR""","""N""",null,"""BASELINE, ANY DAY PRIOR TO FIR…",0,2001,1,"""BEFORE""","""NIDA-CTN-0001""","""MH""","""SCREENING""",31,16,"""ALLERGIES""","""N""",null,"""BASELINE, ANY DAY PRIOR TO FIR…",0,2001,1,"""BEFORE"""
"""NIDA-CTN-0001""","""MH""","""01_000579""","""SCREENING""",6,3,"""CARDIOVASCULAR""","""N""",null,"""BASELINE, ANY DAY PRIOR TO FIR…",0,2001,1,"""DURING/AFTER""","""NIDA-CTN-0001""","""MH""","""SCREENING""",31,16,"""ALLERGIES""","""N""",null,"""BASELINE, ANY DAY PRIOR TO FIR…",0,2001,1,"""BEFORE"""
"""NIDA-CTN-0001""","""MH""","""01_000579""","""SCREENING""",1,1,"""DERMATOLOGICAL""","""N""",null,"""BASELINE, ANY DAY PRIOR TO FIR…",0,2001,1,"""BEFORE""","""NIDA-CTN-0001""","""MH""","""SCREENING""",31,16,"""ALLERGIES""","""N""",null,"""BASELINE, ANY DAY PRIOR TO FIR…",0,2001,1,"""BEFORE"""
"""NIDA-CTN-0001""","""MH""","""01_000579""","""SCREENING""",2,1,"""DERMATOLOGICAL""","""N""",null,"""BASELINE, ANY DAY PRIOR TO FIR…",0,2001,1,"""DURING/AFTER""","""NIDA-CTN-0001""","""MH""","""SCREENING""",31,16,"""ALLERGIES""","""N""",null,"""BASELINE, ANY DAY PRIOR TO FIR…",0,2001,1,"""BEFORE"""
"""NIDA-CTN-0001""","""MH""","""01_000579""","""SCREENING""",23,12,"""ENDOCRINE""","""N""",null,"""BASELINE, ANY DAY PRIOR TO FIR…",0,2001,1,"""BEFORE""","""NIDA-CTN-0001""","""MH""","""SCREENING""",31,16,"""ALLERGIES""","""N""",null,"""BASELINE, ANY DAY PRIOR TO FIR…",0,2001,1,"""BEFORE"""
"""NIDA-CTN-0001""","""MH""","""01_000579""","""SCREENING""",24,12,"""ENDOCRINE""","""N""",null,"""BASELINE, ANY DAY PRIOR TO FIR…",0,2001,1,"""DURING/AFTER""","""NIDA-CTN-0001""","""MH""","""SCREENING""",31,16,"""ALLERGIES""","""N""",null,"""BASELINE, ANY DAY PRIOR TO FIR…",0,2001,1,"""BEFORE"""
"""NIDA-CTN-0001""","""MH""","""01_000579""","""SCREENING""",3,2,"""EYES, EARS, NOSE, AND THROAT""","""N""",null,"""BASELINE, ANY DAY PRIOR TO FIR…",0,2001,1,"""BEFORE""","""NIDA-CTN-0001""","""MH""","""SCREENING""",31,16,"""ALLERGIES""","""N""",null,"""BASELINE, ANY DAY PRIOR TO FIR…",0,2001,1,"""BEFORE"""


In [ ]:
mh.join(
    dm,
    on='USUBJID',
    how='inner',
).shape

(67, 19)

In [ ]:
mh.join(
    dm,
    on='USUBJID',
    how='full',
).shape

(12412, 20)

In [ ]:
###  1. CHARGEMENT ET CONCATÉNATION DES FICHIER
vs = pl.concat([pl.read_csv("vs1.csv"), pl.read_csv("vs2.csv")])
lb = pl.concat([pl.read_csv("lb1.csv"), pl.read_csv("lb2.csv")])
sc = pl.concat([pl.read_csv("sc1.csv"), pl.read_csv("sc2.csv")])

In [ ]:
#Ici les colonnes du fichier qs1 et qs2 ne sont pas pareils on doit donc forcer les types
overrides = {
    "QSEVAL": pl.String,
    "QSORRES": pl.String,
    "QSORRESU": pl.String,
    "QSSTRESC": pl.String
}

# Chargement avec les overrides appliqués
# L'argument 'ignore_errors=True' est ajouté par sécurité
qs1 = pl.read_csv("qs1.csv", schema_overrides=overrides, ignore_errors=True)
qs2 = pl.read_csv("qs2.csv", schema_overrides=overrides, ignore_errors=True)

##  STANDARDISATION ET FUSION

# Harmonisation des noms de colonnes
if "QSORRESU" in qs1.columns and "QSORRES" not in qs1.columns:
    qs1 = qs1.rename({"QSORRESU": "QSORRES"})

if "QSSTRESC" in qs2.columns and "QSORRES" not in qs2.columns:
    qs2 = qs2.rename({"QSSTRESC": "QSORRES"})

# Sélection et fusion
qs1 = qs1.select(["USUBJID", "QSORRES"])
qs2 = qs2.select(["USUBJID", "QSORRES"])
qs = pl.concat([qs1, qs2])

print(f"Chargement QS réussi. Total lignes : {qs.height}")

Chargement QS réussi. Total lignes : 314942


In [ ]:
##  1. DÉFINITION DE LA VARIABLE MANQUANTE
SCREEN_FAILURE_CODE = "Scrnfail"

##  2. FILTRAGE ET SÉLECTION
dm_filtered = dm.filter(
    (pl.col("ARMCD") != SCREEN_FAILURE_CODE) & (pl.col("ARMCD").is_not_null())
)

dm_clean = dm_filtered.select(
    "USUBJID",
    pl.col("ARMCD").alias("ARM"),
    "AGE", "SEX", "RACE", "ETHNIC"
)

# Vérification
print(dm_clean.head())

shape: (2, 6)
┌───────────┬──────┬───────────┬─────┬─────────────────────────────────┬────────┐
│ USUBJID   ┆ ARM  ┆ AGE       ┆ SEX ┆ RACE                            ┆ ETHNIC │
│ ---       ┆ ---  ┆ ---       ┆ --- ┆ ---                             ┆ ---    │
│ str       ┆ str  ┆ f64       ┆ str ┆ str                             ┆ str    │
╞═══════════╪══════╪═══════════╪═════╪═════════════════════════════════╪════════╡
│ 01_000579 ┆ CLON ┆ 27.282683 ┆ F   ┆ WHITE                           ┆ null   │
│ 01_001362 ┆ CLON ┆ 41.429158 ┆ F   ┆ BLACK, AFRICAN AMERICAN, OR NE… ┆ null   │
└───────────┴──────┴───────────┴─────┴─────────────────────────────────┴────────┘


In [ ]:
###  3. PRÉPARATION DU DOMAINE SC (Caractéristiques supplémentaires)
sc_baseline = sc.filter(
    pl.col("SCTESTCD").is_in(["MARITAL", "EDUCYRS", "EMPLOY3Y"])
).select(
    "USUBJID", "SCTESTCD", "SCSTRESC", "SCSTRESN"
).unique(subset=["USUBJID", "SCTESTCD"], keep="first")

sc_wide = sc_baseline.pivot(
    index="USUBJID",
    on="SCTESTCD", # <--- On remplace 'columns' par 'on' car fonction plus récente
    values=["SCSTRESC", "SCSTRESN"],
    aggregate_function="first"
)

sc_clean = sc_wide.with_columns(
    # Recodage MARITAL: Accompagné vs Non (Utilise SCSTRESC)
    MARITAL_GROUP = pl.when(
        (pl.col("SCSTRESC_MARITAL") == "LEGALLY MARRIED") | (pl.col("SCSTRESC_MARITAL") == "LIVING WITH PARTNER-COHABITATING")
    ).then(pl.lit("Accompagné")).otherwise(pl.lit("Non Accompagné")),

    # Recodage EMPLOYMENT (Utilise SCSTRESC)
    EMPLOYMENT_GROUP = pl.when(
        (pl.col("SCSTRESC_EMPLOY3Y") == "FULL TIME (35+ HRS/WK)") | (pl.col("SCSTRESC_EMPLOY3Y") == "PART TIME (REGULAR HOURS)")
    ).then(pl.lit("Régulier/Temps plein")).otherwise(pl.lit("Autre/Irrégulier")),

    # EDUCYRS (Utilise SCSTRESN)
    EDUCYRS = pl.col("SCSTRESN_EDUCYRS").cast(pl.Float64, strict=False)

).select("USUBJID", "MARITAL_GROUP", "EMPLOYMENT_GROUP", "EDUCYRS")


In [ ]:
### 3. PRÉPARATION DU DOMAINE VS (Signes Vitaux)

# Filtrer visites de base / screening
vs_baseline = vs.filter(
    (pl.col("VISITNUM") == 0) | (pl.col("EPOCH") == "SCREENING")
)

## Taille (HEIGHT)
vs_height = vs_baseline.filter(pl.col("VSTESTCD") == "HEIGHT").select(
    "USUBJID",
    (pl.col("VSSTRESN") * 0.0254).alias("HEIGHT_M")  # pouces -> mètres
).group_by("USUBJID").agg([pl.first("HEIGHT_M").alias("HEIGHT_M")])

## Poids (WEIGHT)
vs_weight = vs_baseline.filter(pl.col("VSTESTCD") == "WEIGHT").select(
    "USUBJID",
    (pl.col("VSSTRESN") / 2.20462).alias("WEIGHT_KG")  # lbs -> kg
).group_by("USUBJID").agg([pl.first("WEIGHT_KG").alias("WEIGHT_KG")])

##  Autres signes vitaux (SBP, DBP, PULSE, RESP, TEMP)
vs_other = vs_baseline.filter(
    pl.col("VSTESTCD").is_in(["SBP", "DBP", "PULSE", "RESP", "TEMP"])
).pivot(
    index="USUBJID",
    on="VSTESTCD",
    values="VSSTRESN",
    aggregate_function="first"
).with_columns(
    ## Conversion de TEMP (°F) en TEMP_C (°C)
    TEMP_C = ((pl.col("TEMP") - 32) / 1.8)
).select(
    pl.exclude("TEMP") # Suppression de la colonne TEMP (°F) brute
).rename({"SBP": "SBP", "DBP": "DBP", "PULSE": "PULSE", "RESP": "RESP"})


#  Fusion VS (Height, Weight, autres signes vitaux) et calcul BMI
vs_clean = vs_height.join(vs_weight, on="USUBJID", how="left")\
                     .join(vs_other, on="USUBJID", how="left")\
                     .with_columns([
                         (pl.col("WEIGHT_KG") / (pl.col("HEIGHT_M")**2)).alias("BMI")
                     ])


In [ ]:
### 4. PRÉPARATION DU DOMAINE MH (Antécédents Médicaux)

## 4a. Agréger MH pour trouver les sujets avec un antécédent pertinent
mh_flag = mh.filter(
    (pl.col("MHOCCUR") != "N") & (pl.col("MHTERM").is_not_null())
).select("USUBJID").unique()\
.with_columns(
    # Ajouter un flag temporaire à vérifier lors de la jointure
    is_histo = pl.lit(True)
)

#  Joindre à DM et créer le flag binaire
mh_flag_final = dm_clean.select("USUBJID").join(
    mh_flag, on="USUBJID", how="left"
).with_columns(
    #  Vérifier la colonne is_histo au lieu de USUBJID_right
    pl.when(pl.col("is_histo").is_not_null())
      .then(pl.lit("Oui"))
      .otherwise(pl.lit("Non"))
      .alias("HISTO_PERTINENT")
).select("USUBJID", "HISTO_PERTINENT")

In [ ]:
###  5. PRÉPARATION DU DOMAINE MH (Antécédents Médicaux)
#  Ajout du flag is_histo pour la vérification après jointure
mh_flag = mh.filter(
    (pl.col("MHOCCUR") != "N") & (pl.col("MHTERM").is_not_null())
).select("USUBJID").unique().with_columns(is_histo = pl.lit(True))

mh_flag_final = dm_clean.select("USUBJID").join(
    mh_flag, on="USUBJID", how="left"
).with_columns(
    # Vérification du flag 'is_histo' au lieu de 'USUBJID_right'
    pl.when(pl.col("is_histo").is_not_null()).then(pl.lit("Oui")).otherwise(pl.lit("Non"))
    .alias("HISTO_PERTINENT")
).select("USUBJID", "HISTO_PERTINENT")


In [ ]:
###  6. PRÉPARATION DES DOMAINES LB & QS (Labo et Scores)

## Extraction des tests urinaires directement de LB
lb_tests = lb.filter(
    (pl.col("VISITNUM") == 0) | (pl.col("EPOCH") == "SCREENING")
).filter(
    ##  Utilisation de METHADON et MORPHINE
    pl.col("LBTESTCD").is_in(["METHADON", "MORPHINE"])
).select(
    "USUBJID", pl.col("LBTESTCD").alias("TEST_NAME"), pl.col("LBORRES").alias("TEST_RESULT")
).pivot(
    index="USUBJID", on="TEST_NAME", values="TEST_RESULT", aggregate_function="first"
).select(
    "USUBJID",
    ##  Sélection des colonnes avec les codes exacts
    pl.col("METHADON").alias("Test_Methadone"),
    pl.col("MORPHINE").alias("Test_Morphine")
)

# Extraction du score VAS à l'inclusion (QS)
qs_baseline = qs.select("USUBJID", pl.col("QSORRES").alias("VAS_Score")).unique(subset=["USUBJID"], keep="first")

# Fusion des Labos et des Scores
lb_qs_clean = lb_tests.join(qs_baseline, on="USUBJID", how="left")

In [ ]:
###  7. FUSION FINALE DES DOMAINES POUR FAIRE LE TABLEAU
df_analysis = dm_clean.join(
    sc_clean, on="USUBJID", how="left"
).join(
    vs_clean, on="USUBJID", how="left"
).join(
    mh_flag_final, on="USUBJID", how="left"
).join(
    lb_qs_clean, on="USUBJID", how="left"
).to_pandas()
#On a fait une jointure de sc,vs,mh et lb
#On affiche le dataframe
print(f"DataFrame d'analyse final créé. Nombre de sujets: {len(df_analysis)}")

DataFrame d'analyse final créé. Nombre de sujets: 2


In [ ]:
###  8. GÉNÉRATION DU TABLEAU 1 AVEC TABLEONE

columns = [
    'AGE', 'SEX', 'RACE', 'ETHNIC',
    'EDUCYRS', 'MARITAL_GROUP', 'EMPLOYMENT_GROUP',
    'HEIGHT_M', 'WEIGHT_KG', 'BMI',
    'SBP', 'DBP', 'PULSE', 'RESP', 'TEMP_C',
    'VAS_Score',
    'HISTO_PERTINENT',
    'Test_Methadone', 'Test_Morphine'
]

categorical = [
    'SEX', 'RACE', 'ETHNIC',
    'MARITAL_GROUP', 'EMPLOYMENT_GROUP',
    'HISTO_PERTINENT', 'Test_Methadone', 'Test_Morphine'
]

nonnormal = [
    'AGE', 'EDUCYRS', 'BMI', 'VAS_Score'
]

groupby = 'ARM'

# Création de l'objet TableOne
table = TableOne(
    df_analysis,
    columns=columns,
    categorical=categorical,
    groupby=groupby,
    nonnormal=nonnormal,
    pval=False, # <- CORRECTION: Désactivation des tests statistiques
    decimals={
        'AGE': 1, 'HEIGHT_M': 2, 'WEIGHT_KG': 1, 'BMI': 2,
        'SBP': 0, 'DBP': 0, 'PULSE': 0, 'RESP': 0, 'TEMP_C': 1,
        'VAS_Score': 1, 'EDUCYRS': 0
    },
    label_suffix=True,
    rename = {
        'AGE': 'Âge (années)', 'SEX': 'Sexe', 'RACE': 'Origine Ethnique',
        'ETHNIC': 'Appartenance Ethnique',
        'EDUCYRS': "Années d'étude", 'MARITAL_GROUP': 'Statut Marital', 'EMPLOYMENT_GROUP': 'Statut Professionnel',
        'HEIGHT_M': 'Taille (m)', 'WEIGHT_KG': 'Poids (kg)', 'BMI': 'IMC (kg/m$^2$)',
        'SBP': 'PAS (mmHg)', 'DBP': 'PAD (mmHg)',
        'PULSE': 'Fréquence Cardiaque (bpm)', 'RESP': 'Fréquence Respiratoire (/min)',
        'TEMP_C': 'Température corporelle (°C)',
        'VAS_Score': 'Score Analogique Visuel (début)',
        'HISTO_PERTINENT': 'Antécédent Médical Pertinent',
        'Test_Methadone': 'Test Methadone (Positif)', 'Test_Morphine': 'Test Morphine (Positif)',
    }
)

# Affichage du Tableau 1
print("\n--- TABLEAU 1: CARACTÉRISTIQUES DÉMOGRAPHIQUES ET CLINIQUES DE BASE (FINAL) ---") #Le Titre du Tableau
print(table.tabulate(tablefmt="markdown")) #Afficher le tableau


--- TABLEAU 1: CARACTÉRISTIQUES DÉMOGRAPHIQUES ET CLINIQUES DE BASE (FINAL) ---
                                                                                    Missing    Overall              CLON
-----------------------------------------------  ---------------------------------  ---------  -------------------  -------------------
n                                                                                              2                    2
Âge (années), median [Q1,Q3]                                                        0          34.4 [30.8,37.9]     34.4 [30.8,37.9]
Sexe, n (%)                                      F                                             2 (100.0)            2 (100.0)
Origine Ethnique, n (%)                          BLACK, AFRICAN AMERICAN, OR NEGRO             1 (50.0)             1 (50.0)
                                                 WHITE                                         1 (50.0)             1 (50.0)
Appartenance Ethnique, n (%)       